# Calibrating and Stress-Testing the Founder-Departure Diffusion Result\n\nThis notebook demonstrates the **evaluation script** (`eval.py`) for the founder-authority-diffusion-vs-survival experiment.\n\nThe original evaluation is a two-stage pipeline:\n\n- **Stage A (calibration gate):** recomputes Avelino et al. (ESEM 2019)'s three headline aggregate statistics -- Truck-Factor-Detachment-Departure (TFDD) incidence rate, share of TFDDs at truck-factor 1, and overall 18-month survival rate -- over the dataset's raw commit event log, each with a 95% Wilson confidence interval and a PASS/FLAG_DEVIATION status, plus a snapshot-null Cohen's d replication.\n- **Stage B (robustness checks):** five checks on the main experiment's founder-only-TFDD diffusion-vs-survival finding, including window-boundary sensitivity, founder-identification-heuristic sensitivity, an age-at-TFDD confound check, matched-pairs bucket-definition sensitivity, and a placebo/permutation test.\n\nThe full `eval.py` also re-runs the upstream experiment's `method.py` on raw git history (which needs the full dataset dependency and is not portable to a notebook). This demo instead loads a **curated snapshot of the same intermediate values `eval.py` produces internally** (the 8 detected TFDD events from the real run, plus the Stage B check outputs) and re-executes `eval.py`'s own statistical helper functions (Wilson CI, Cohen's d, bootstrap CI) on that data -- exactly the same code, just applied to a saved slice of the pipeline's output rather than to freshly-mined git history.

In [ ]:
import subprocess, sys\ndef _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])\n\n# statsmodels -- NOT pre-installed on Colab, always install\n_pip('statsmodels==0.14.6')\n\n# numpy, pandas, scipy, matplotlib -- pre-installed on Colab, install locally only\nif 'google.colab' not in sys.modules:\n    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import json\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\nfrom statsmodels.stats.multitest import multipletests\nimport matplotlib.pyplot as plt

## Load data\n\n`mini_demo_data.json` is a curated snapshot of `eval.py`'s own intermediate output from a real run: the 8 TFDD (Truck-Factor-Detachment-Departure) events it detected across the 15-repo corpus, plus the Stage B robustness-check results and final scoring. We try the GitHub-hosted copy first, falling back to the local file (works both in Colab and locally).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-24ffbe-pre-departure-bus-factor-diffusion/main/round-1/evaluation-1/demo/mini_demo_data.json"\nimport json, os\n\ndef load_data():\n    try:\n        import urllib.request\n        with urllib.request.urlopen(GITHUB_DATA_URL) as response:\n            return json.loads(response.read().decode())\n    except Exception: pass\n    if os.path.exists("mini_demo_data.json"):\n        with open("mini_demo_data.json") as f: return json.load(f)\n    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()\nprint("n_corpus:", data["n_corpus"])\nprint("n_tfdd_all:", data["n_tfdd_all"])\nprint("n TFDD event records loaded:", len(data["all_tfdd_events_detail"]))

## Config\n\nAll tunable parameters from the original `eval.py`, copied as-is. `N_BOOTSTRAP` and `N_PERMUTATIONS` are set to their absolute-minimum demo values here (they get scaled up in the next cell if time permits) -- the original values are commented alongside each.

In [ ]:
RNG_SEED = 20260820\nN_BOOTSTRAP = 200   # original: 2000\nN_PERMUTATIONS = 60  # original: N_PERMUTATIONS = 60 (already capped in eval.py -- see comment there)

## Statistical helpers\n\nThese are `eval.py`'s own small stat helper functions, copied unchanged: a Wilson score 95% CI for a binomial proportion, a generic bootstrap CI, Cohen's d, and a Benjamini-Hochberg multiple-testing adjustment.

In [ ]:
from typing import Optional\n\ndef wilson_ci(k: int, n: int, z: float = 1.959963985) -> tuple[Optional[float], Optional[float], Optional[float]]:\n    """Wilson score 95% CI for a binomial proportion. Returns (phat, lo, hi)."""\n    if n == 0:\n        return None, None, None\n    phat = k / n\n    denom = 1 + z**2 / n\n    center = phat + z**2 / (2 * n)\n    half = z * np.sqrt(phat * (1 - phat) / n + z**2 / (4 * n**2))\n    lo = (center - half) / denom\n    hi = (center + half) / denom\n    return float(phat), float(max(0.0, lo)), float(min(1.0, hi))\n\n\ndef bootstrap_ci(values: np.ndarray, stat_fn, n_boot: int = N_BOOTSTRAP, seed: int = RNG_SEED) -> dict:\n    rng = np.random.default_rng(seed)\n    values = np.asarray(values)\n    if len(values) == 0:\n        return {"point": None, "ci_95": [None, None], "n_boot": 0}\n    point = float(stat_fn(values))\n    boots = []\n    for _ in range(n_boot):\n        sample = rng.choice(values, size=len(values), replace=True)\n        try:\n            boots.append(float(stat_fn(sample)))\n        except Exception:\n            continue\n    if not boots:\n        return {"point": point, "ci_95": [None, None], "n_boot": 0}\n    boots = np.array(boots)\n    return {\n        "point": point,\n        "ci_95": [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))],\n        "n_boot": len(boots),\n    }\n\n\ndef cohens_d(a: np.ndarray, b: np.ndarray) -> Optional[float]:\n    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)\n    a, b = a[~np.isnan(a)], b[~np.isnan(b)]\n    if len(a) < 2 or len(b) < 2:\n        return None\n    na, nb = len(a), len(b)\n    pooled_sd = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))\n    if pooled_sd == 0:\n        return None\n    return float((a.mean() - b.mean()) / pooled_sd)\n\n\ndef bh_adjust(pvals: dict) -> dict:\n    keys = list(pvals.keys())\n    vals = [pvals[k] for k in keys]\n    if not vals:\n        return {}\n    _, p_bh, _, _ = multipletests(vals, method="fdr_bh")\n    return dict(zip(keys, [float(p) for p in p_bh]))

## Stage A: calibration checks 1-4\n\nRecompute `eval.py`'s `stage_a_calibration` checks 1-4 directly from the loaded TFDD event records (`tfdd_events` = `data["all_tfdd_events_detail"]`), against the corpus size `n_corpus` and `n_tfdd` = `n_tfdd_all` -- same logic as in `stage_a_calibration()`, just applied to the saved events instead of freshly re-detected ones.

In [ ]:
tfdd_events = data["all_tfdd_events_detail"]\nn_corpus = data["n_corpus"]\nn_tfdd = data["n_tfdd_all"]\n\n# --- check 1: TFDD incidence rate vs Avelino 16% (315/1932) ---\nphat, lo, hi = wilson_ci(n_tfdd, n_corpus) if n_corpus else (None, None, None)\ncheck1 = {\n    "metric": "tfdd_incidence_rate",\n    "reimplemented_rate": phat, "ci_95": [lo, hi], "n_corpus": n_corpus, "n_tfdd": n_tfdd,\n    "avelino_reference": 315 / 1932, "avelino_n": "315/1932",\n    "abs_deviation": (abs(phat - 315 / 1932) if phat is not None else None),\n    "rel_deviation": (abs(phat - 315 / 1932) / (315 / 1932) if phat is not None else None),\n    "status": ("PASS" if (phat is not None and lo <= 315 / 1932 <= hi) else "FLAG_DEVIATION"),\n}\n\n# --- check 2: TF=1 share among TFDDs vs Avelino 66% ---\nn_tf1 = sum(1 for r in tfdd_events if r["tf_size"] == 1)\nphat2, lo2, hi2 = wilson_ci(n_tf1, n_tfdd) if n_tfdd else (None, None, None)\ncheck2 = {\n    "metric": "tf1_share_of_tfdd",\n    "reimplemented_rate": phat2, "ci_95": [lo2, hi2], "n_tfdd": n_tfdd, "n_tf1": n_tf1,\n    "avelino_reference": 0.66,\n    "abs_deviation": (abs(phat2 - 0.66) if phat2 is not None else None),\n    "rel_deviation": (abs(phat2 - 0.66) / 0.66 if phat2 is not None else None),\n    "status": ("PASS" if (phat2 is not None and lo2 <= 0.66 <= hi2) else "FLAG_DEVIATION"),\n}\n\n# --- check 3: overall 18mo survival rate among ALL TFDDs vs Avelino 41% (128/315) ---\nn_survived = sum(1 for r in tfdd_events if r.get("survived_binary") == 1)\nphat3, lo3, hi3 = wilson_ci(n_survived, n_tfdd) if n_tfdd else (None, None, None)\ncheck3 = {\n    "metric": "overall_18mo_survival_rate",\n    "reimplemented_rate": phat3, "ci_95": [lo3, hi3], "n_tfdd": n_tfdd, "n_survived": n_survived,\n    "avelino_reference": 128 / 315, "avelino_n": "128/315",\n    "abs_deviation": (abs(phat3 - 128 / 315) if phat3 is not None else None),\n    "rel_deviation": (abs(phat3 - 128 / 315) / (128 / 315) if phat3 is not None else None),\n    "status": ("PASS" if (phat3 is not None and lo3 <= 128 / 315 <= hi3) else "FLAG_DEVIATION"),\n}\n\n# --- check 4: snapshot-null Cohen's d replication (Avelino: 0.13-0.26, negligible-small) ---\nsurv = [r for r in tfdd_events if r.get("survived_binary") == 1]\nnonsurv = [r for r in tfdd_events if r.get("survived_binary") == 0]\nd_results = {}\nfor feat in ["developers_at_tfdd", "commits_at_tfdd", "files_at_tfdd"]:\n    a = np.array([r[feat] for r in surv if r.get(feat) is not None], dtype=float)\n    b = np.array([r[feat] for r in nonsurv if r.get(feat) is not None], dtype=float)\n    d_results[feat] = cohens_d(a, b)\nvalid_ds = [v for v in d_results.values() if v is not None]\nd_in_range = all(0.0 <= abs(v) <= 0.5 for v in valid_ds) if valid_ds else None\ncheck4 = {\n    "metric": "snapshot_null_cohens_d",\n    "cohens_d_per_feature": d_results,\n    "avelino_reference_range": [0.13, 0.26],\n    "n_survivors": len(surv), "n_nonsurvivors": len(nonsurv),\n    "status": ("PASS" if d_in_range else ("FLAG_DEVIATION" if valid_ds else "UNAVAILABLE_INSUFFICIENT_N")),\n}\n\nfor c in (check1, check2, check3, check4):\n    print(c["metric"], "->", c["status"])

## Stage B: robustness checks (loaded from the saved run)\n\nStage B's checks 6-10 (window-boundary sensitivity, founder-ID sensitivity, age confound, bucket-definition sensitivity, permutation test) operate on the raw per-commit event log of each repo, which is not shipped in this small demo dataset. We load `eval.py`'s own outputs for those checks from the saved run instead, and use the `bh_adjust` helper above to show how the BH-adjusted p-values in check 6 were produced.

In [ ]:
check6 = data["check6_window_boundary_sensitivity"]\ncheck8 = data["check8_age_confound"]\ncheck10 = data["check10_placebo_permutation"]\nfinal_scoring = data["final_scoring"]\n\n# demonstrate bh_adjust on the raw founder_share_pre p-values from check6's variants\np_raw = {v["variant"]: v["fit"]["logistic"]["p_founder_share_pre"] for v in check6["variants"]\n         if v.get("fit", {}).get("logistic", {}).get("p_founder_share_pre") == v.get("fit", {}).get("logistic", {}).get("p_founder_share_pre")}\np_bh = bh_adjust(p_raw)\nprint("check6 sign_stable_across_variants:", check6["sign_stable_across_variants"])\nprint("check6 raw p-values:", p_raw)\nprint("check6 BH-adjusted p-values (recomputed here):", p_bh)\nprint()\nprint("check8 diffusion_coef_survives_age_control:", check8.get("diffusion_coef_survives_age_control"))\nprint("check10 permutation_p_value_pooled:", check10.get("permutation_p_value_pooled"))

## Results\n\nSummary table of Stage A checks (with 95% CIs vs the Avelino et al. reference values) and a plot comparing the reimplemented rates to the reference rates, plus the final scoring verdict.

In [ ]:
summary_rows = []\nfor c in (check1, check2, check3):\n    summary_rows.append({\n        "metric": c["metric"],\n        "reimplemented": round(c["reimplemented_rate"], 3) if c["reimplemented_rate"] is not None else None,\n        "ci_95_lo": round(c["ci_95"][0], 3) if c["ci_95"][0] is not None else None,\n        "ci_95_hi": round(c["ci_95"][1], 3) if c["ci_95"][1] is not None else None,\n        "avelino_reference": round(c["avelino_reference"], 3),\n        "status": c["status"],\n    })\nsummary_df = pd.DataFrame(summary_rows)\nprint(summary_df.to_string(index=False))\nprint()\ngate_status = "FLAG_DEVIATION" if any(c["status"] == "FLAG_DEVIATION" for c in (check1, check2, check3, check4)) else "PASS"\nprint("Stage A gate_status:", gate_status)\nprint("Overall verdict (from saved run):", final_scoring["overall_verdict"])\nprint("power_caveat:", final_scoring["power_caveat"])\n\nfig, ax = plt.subplots(figsize=(7, 4))\nx = np.arange(len(summary_df))\nwidth = 0.35\nax.bar(x - width/2, summary_df["reimplemented"], width, label="reimplemented (this corpus)",\n       yerr=[summary_df["reimplemented"] - summary_df["ci_95_lo"], summary_df["ci_95_hi"] - summary_df["reimplemented"]],\n       capsize=4, color="#4C72B0")\nax.bar(x + width/2, summary_df["avelino_reference"], width, label="Avelino et al. reference", color="#DD8452")\nax.set_xticks(x)\nax.set_xticklabels(summary_df["metric"], rotation=20, ha="right")\nax.set_ylabel("rate")\nax.set_title("Stage A calibration: reimplemented vs. reference rates")\nax.legend()\nplt.tight_layout()\nplt.show()